# Naive RAG with Pinecone

### Why RAG exists

Ask a model a question directly and you run into three problems. Its knowledge is **stale** — frozen at whatever point it was trained, so it cannot know what happened this morning. It has **no access to private data** — your own documents were never part of its training, so it has no way to see them. And when it doesn't know something, it often **hallucinates** — stating an invented answer with total confidence.

Retrieval-Augmented Generation (RAG) fixes this with a simple idea: **look up relevant material first, then answer.** Instead of a closed-book exam where the model answers purely from memory, it's an open-book exam — the source material is sitting right in front of it, and the answer is grounded in text you can actually point at.

### The pipeline, end to end

There are two separate pipelines here, and keeping them separate in your head is the single most useful thing to take from this notebook:

- **Ingestion (ahead of time, runs once per document):** Load → Chunk → Embed → store in a vector database
- **Retrieval (runs on every question):** Embed the question → retrieve the top-k closest chunks → paste them into a prompt → generate the answer

This notebook builds both halves, using **Pinecone** as the vector database. Run cells with Shift+Enter, top to bottom.

## Setup

Nothing conceptual here — just installing libraries and providing two API keys: one for **OpenAI** (to generate embeddings and answers) and one for **Pinecone** (the vector database that will store our chunks). Get a free Pinecone key at https://app.pinecone.io if you don't have one.

The uninstall step below exists because an older package, `pinecone-client`, used to share the same import path as the current `pinecone` package. If both end up installed at once they conflict, so we remove the old one first to be safe.

In [2]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")
os.environ["PINECONE_API_KEY"] = getpass("Paste your Pinecone API key: ")

Paste your OpenAI API key:  ········
Paste your Pinecone API key:  ········


## The document we'll work with

Every technique in this notebook needs something to chew on. Run the cell below to generate a small three-page sample PDF, or upload your own via the folder icon on the left sidebar and point Step 1 at it instead.

In [3]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

pages = [
    "Library Handbook\n\n"
    "Hours of operation: The library opens at 8:00 AM on weekdays and 10:00 AM on weekends. "
    "It closes at 9:00 PM every day.\n\n"
    "Borrowing: Students may borrow up to 8 books for 3 weeks. Faculty may borrow up to 20 books for a semester.",

    "Budget Committee Notes\n\n"
    "The budget was approved in March after debate in committee. The vote was 14 to 3. "
    "The approved amount was $42,000, allocated to periodicals and digital subscriptions.",

    "Study Room Policy\n\n"
    "Group study rooms can be reserved online for up to 2 hours. "
    "Quiet study areas on the third floor are first come, first served. Food is not permitted; covered drinks are allowed.",
]

c = canvas.Canvas("handbook.pdf", pagesize=letter)
for page_text in pages:
    y = 750
    for line in page_text.split("\n"):
        c.drawString(72, y, line)
        y -= 18
    c.showPage()
c.save()
print("Wrote handbook.pdf")

Wrote handbook.pdf


## Step 1 — Load

The first job is just getting the document's raw text into memory. `PyPDFLoader` reads the PDF and returns one LangChain `Document` per page, each carrying its own text plus metadata (like which page it came from) — that metadata is what lets you trace an answer back to its source later.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("handbook.pdf")
documents = loader.load()

print(len(documents))
print(documents[0].metadata)

/tmp/ipykernel_950/1772006447.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


3
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-10-31T15:09:08+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-10-31T15:09:08+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': 'handbook.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


## Step 2 — Chunking

### Why split documents at all?

You cannot hand a whole document to the model for every question — it's slow, it's expensive, and irrelevant text drowns out the part that actually matters. So documents get split into smaller pieces ahead of time, and each piece is retrieved independently.

This creates the trade-off that defines everything about chunking: **small chunks match precisely but lose context. Large chunks keep context but match imprecisely.** Every technique below is an attempt to get both sides of that trade-off at once.

We'll compare four techniques on the same document.

### 1. Fixed-size / character

Split every N characters, on a single separator, no matter what's actually at that boundary. It's fast and simple, but it's also the worst quality — it will cut a sentence, or a fact, straight down the middle without knowing it did so.

In [10]:
from langchain_text_splitters import CharacterTextSplitter

fixed_splitter = CharacterTextSplitter(separator="", chunk_size=500, chunk_overlap=50)
fixed_chunks = fixed_splitter.split_documents(documents)

print([len(c.page_content) for c in fixed_chunks])
for i, chunk in enumerate(fixed_chunks):
    print(f"--- Chunk {i} (len={len(chunk.page_content)}) ---")
    print(chunk.page_content)
    print()

[243, 189, 195]
--- Chunk 0 (len=243) ---
Library Handbook
Hours of operation: The library opens at 8:00 AM on weekdays and 10:00 AM on weekends. It closes at 9:00 PM every day.
Borrowing: Students may borrow up to 8 books for 3 weeks. Faculty may borrow up to 20 books for a semester.

--- Chunk 1 (len=189) ---
Budget Committee Notes
The budget was approved in March after debate in committee. The vote was 14 to 3. The approved amount was $42,000, allocated to periodicals and digital subscriptions.

--- Chunk 2 (len=195) ---
Study Room Policy
Group study rooms can be reserved online for up to 2 hours. Quiet study areas on the third floor are first come, first served. Food is not permitted; covered drinks are allowed.



### 2. Recursive character — the sensible default

Instead of one rigid separator, this tries paragraph breaks first, then line breaks, then words, then individual characters — dropping down to a smaller separator only when a piece is still too big. It keeps related text together far more often than fixed-size does. The one gap: if a single paragraph is longer than the chunk limit, it stays oversized — there's no further fallback.

This is the technique to reach for by default. Only move to something fancier once your retrieval results — not your intuition — tell you it's the bottleneck.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""], chunk_size=500, chunk_overlap=50
)
recursive_chunks = recursive_splitter.split_documents(documents)

print([len(c.page_content) for c in recursive_chunks])
for i, chunk in enumerate(recursive_chunks):
    print(f"--- Chunk {i} (len={len(chunk.page_content)}) ---")
    print(chunk.page_content)
    print()

[243, 189, 195]
--- Chunk 0 (len=243) ---
Library Handbook
Hours of operation: The library opens at 8:00 AM on weekdays and 10:00 AM on weekends. It closes at 9:00 PM every day.
Borrowing: Students may borrow up to 8 books for 3 weeks. Faculty may borrow up to 20 books for a semester.

--- Chunk 1 (len=189) ---
Budget Committee Notes
The budget was approved in March after debate in committee. The vote was 14 to 3. The approved amount was $42,000, allocated to periodicals and digital subscriptions.

--- Chunk 2 (len=195) ---
Study Room Policy
Group study rooms can be reserved online for up to 2 hours. Quiet study areas on the third floor are first come, first served. Food is not permitted; covered drinks are allowed.



### 3. Semantic — split where the topic actually shifts

This embeds each sentence, measures how far apart consecutive sentences are in meaning, and splits where the topic genuinely changes — not where a character count happens to land. It keeps context very well, but it costs one embedding call per sentence, so it's noticeably slower. Watch the runtime on this cell versus the two above.

In [5]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

semantic_splitter = SemanticChunker(OpenAIEmbeddings())
semantic_chunks = semantic_splitter.split_documents(documents)

print([len(c.page_content) for c in semantic_chunks])
for i, chunk in enumerate(semantic_chunks):
    print(f"--- Chunk {i} (len={len(chunk.page_content)}) ---")
    print(chunk.page_content)
    print()

/tmp/ipykernel_950/912547020.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


[135, 107, 104, 84, 144, 50]
--- Chunk 0 (len=135) ---
Library Handbook
Hours of operation: The library opens at 8:00 AM on weekdays and 10:00 AM on weekends. It closes at 9:00 PM every day.

--- Chunk 1 (len=107) ---
Borrowing: Students may borrow up to 8 books for 3 weeks. Faculty may borrow up to 20 books for a semester.

--- Chunk 2 (len=104) ---
Budget Committee Notes
The budget was approved in March after debate in committee. The vote was 14 to 3.

--- Chunk 3 (len=84) ---
The approved amount was $42,000, allocated to periodicals and digital subscriptions.

--- Chunk 4 (len=144) ---
Study Room Policy
Group study rooms can be reserved online for up to 2 hours. Quiet study areas on the third floor are first come, first served.

--- Chunk 5 (len=50) ---
Food is not permitted; covered drinks are allowed.



### 4. Structure-aware — split on the document's own headings

Real documents aren't plain text. A naive splitter run over a PDF treats a table as a wall of characters — rows get cut apart, column headers separate from their numbers, and what reaches the model is noise that looks like data. Structure-aware splitting detects the document's actual headings up front and splits along them instead, tracking each chunk's section path as metadata.

PDFs don't carry heading markup the way Markdown does, so we recreate the handbook as Markdown here purely to demonstrate the technique. In production you'd get this structure from a layout-aware parser instead.

In [6]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_text = """
# Library Handbook

## Hours of Operation
The library opens at 8:00 AM on weekdays and 10:00 AM on weekends. It closes at 9:00 PM every day.

## Borrowing
Students may borrow up to 8 books for 3 weeks. Faculty may borrow up to 20 books for a semester.

# Budget Committee Notes

## March Approval
The budget was approved in March after debate in committee. The vote was 14 to 3. The approved amount was $42,000, allocated to periodicals and digital subscriptions.

# Study Room Policy

## Reservations
Group study rooms can be reserved online for up to 2 hours.

## Restrictions
Food is not permitted; covered drinks are allowed.
"""

headers_to_split_on = [("#", "Header 1"), ("##", "Header 2")]
structure_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
structure_chunks = structure_splitter.split_text(markdown_text)

for i, chunk in enumerate(structure_chunks):
    print(f"--- Chunk {i} ---")
    print("Metadata:", chunk.metadata)
    print(chunk.page_content)
    print()

--- Chunk 0 ---
Metadata: {'Header 1': 'Library Handbook', 'Header 2': 'Hours of Operation'}
The library opens at 8:00 AM on weekdays and 10:00 AM on weekends. It closes at 9:00 PM every day.

--- Chunk 1 ---
Metadata: {'Header 1': 'Library Handbook', 'Header 2': 'Borrowing'}
Students may borrow up to 8 books for 3 weeks. Faculty may borrow up to 20 books for a semester.

--- Chunk 2 ---
Metadata: {'Header 1': 'Budget Committee Notes', 'Header 2': 'March Approval'}
The budget was approved in March after debate in committee. The vote was 14 to 3. The approved amount was $42,000, allocated to periodicals and digital subscriptions.

--- Chunk 3 ---
Metadata: {'Header 1': 'Study Room Policy', 'Header 2': 'Reservations'}
Group study rooms can be reserved online for up to 2 hours.

--- Chunk 4 ---
Metadata: {'Header 1': 'Study Room Policy', 'Header 2': 'Restrictions'}
Food is not permitted; covered drinks are allowed.



### Choosing between them

There is no single best technique — only trade-offs.

| Technique | Cost | Keeps context | Use it when |
|---|---|---|---|
| Fixed-size | Lowest | Poorly | Prototyping only |
| Recursive character | Low | Usually | Your default starting point |
| Semantic | High | Very well | Recursive is provably the bottleneck |
| Structure-aware | Low | By section | Real headings, tables, reports |

We'll carry `recursive_chunks` forward for the rest of this notebook — swap the line below to compare a different technique end-to-end.

In [17]:
chunks = semantic_chunks   # try swapping for fixed_chunks or semantic_chunks
print("Using", len(chunks), "chunks going forward")

Using 6 chunks going forward


## Step 3 — Embeddings: turning meaning into numbers

An embedding converts a piece of text into a list of numbers that captures what it *means*. Two sentences about the same idea land close together in that number space, even if they share no words at all — "The cat sat on the mat" and "A feline rested on the rug" have zero words in common but produce nearly identical vectors.

This is what separates retrieval from Ctrl+F: keyword search finds matching characters, embedding search finds matching meaning.

In [7]:
from langchain_openai import OpenAIEmbeddings

embedder = OpenAIEmbeddings(model="text-embedding-3-small")
vector = embedder.embed_query("the cat sat")

print(len(vector))   # 1536 numbers representing this sentence's meaning
print(vector[:5])

1536
[-0.01244354248046875, -0.04449462890625, -0.0173492431640625, 0.007419586181640625, 0.0447998046875]


## Step 4 — Vector store: built for one question

A vector store exists to answer exactly one question fast, at scale: *"given this vector, which stored vectors are closest to it?"*

We're using **Pinecone** — a fully managed vector database. It handles the index for you, which is why the next cell has to create one before anything can be stored in it. Because `text-embedding-3-small` produces 1536-dimensional vectors, the index has to be created with `dimension=1536` to match — this only needs to run once; if the index already exists, the cell just skips creation.

In [8]:
from pinecone import Pinecone, ServerlessSpec
import time

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "naive-rag-classroom"

if index_name not in [i.name for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

print("Index ready:", index_name)

ModuleNotFoundError: No module named 'pinecone'

Now every chunk from Step 2 gets embedded and upserted into that index in one call. This is the last step of the ingestion pipeline — everything from here on happens at query time, on every question.

In [20]:
from langchain_pinecone import PineconeVectorStore

store = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embedder,
    index_name=index_name,
)
print("Stored", len(chunks), "chunks in Pinecone")

Stored 6 chunks in Pinecone


## Step 5 — Retrieve

This is where query time begins. The question gets embedded the exact same way the chunks were, and the store returns the top-k closest matches by meaning — commonly 3 to 5. Notice we're inspecting the retrieved chunks *before* generating an answer: that's deliberate. It's what makes RAG feel inspectable rather than magical — you can see exactly what the model is about to be handed.

In [21]:
question = "When does the library open?"
matches = store.similarity_search(question, k=3)

for m in matches:
    print(m.metadata.get("page"), "-", m.page_content[:80].replace("\n", " "))

0.0 - Library Handbook Hours of operation: The library opens at 8:00 AM on weekdays an
0.0 - Library Handbook Hours of operation: The library opens at 8:00 AM on weekdays an
0.0 - Borrowing: Students may borrow up to 8 books for 3 weeks. Faculty may borrow up 


## Step 6 — Generate

The retrieved chunks get pasted into the prompt alongside the question. The model answers from what's in front of it, not from memory — that's the whole trick behind RAG. The "if the context does not contain the answer, say so" instruction matters: without it, the model will often produce a confident answer anyway, stitched together from the closest chunks it could find, even when none of them actually answer the question.

In [22]:
from langchain_openai import ChatOpenAI

context = "\n\n".join(m.page_content for m in matches)

prompt = f"""Answer the question using only the context below.
If the context does not contain the answer, say so.

Context:
{context}

Question: {question}"""

answer = ChatOpenAI(model="gpt-4o-mini").invoke(prompt)
print(answer.content)

The library opens at 8:00 AM on weekdays and 10:00 AM on weekends.


## Where naive RAG breaks

Worth sitting with before you call this "done":

1. **Similar is not the same as useful.** Top-k returns the closest matches by meaning. Closest is not a guarantee of relevant, and nothing in this pipeline checks.
2. **One shot, no second chances.** Exactly one retrieval, exactly one generation. If the retrieval was bad, the answer is bad — and the system has no way to notice.
3. **Nothing verifies the answer.** The model is given context. It isn't required to use it, and nothing checks whether the final answer is actually supported by it.
4. **What if the answer isn't there at all?** Naive RAG will still confidently produce something, assembled from the closest chunks it could find, even if none of them are right.

Every advanced RAG technique you'll hear about exists to fix one of these four.

## Cleanup (optional)

Free-tier Pinecone accounts have limited indexes/storage. Run this at the end of class if you want to delete the index.

In [ ]:
# pc.delete_index(index_name)
# print("Deleted", index_name)

## Your turn

1. Change `chunks = recursive_chunks` in Step 2 to `fixed_chunks`, re-run Steps 4–6, and compare the final answer's quality.
2. Set `chunk_size=50` in the recursive splitter, re-run, and ask a question whose answer spans a paragraph. Watch it fall apart.
3. Print `matches` for a question the system gets wrong. Was the failure in retrieval, or in generation?
4. Ask something the document genuinely can't answer, with and without the "say so" instruction in the prompt.
5. If a company's documents change every day, how often does the vector store need updating — and what breaks if it isn't?
6. Can you think of a question where the most *similar* chunk and the most *useful* chunk are different things?

In [ ]:
# Scratch space
